# Travel Planner Generator

This notebook implements an AI-powered meta-software development workflow for the DTS114TC coursework. It turns a travel planning business problem into software artefacts, including SDLC documentation, UML diagrams, a Flask API, a travel planning website, and deployment assets.

## Development Methodology

The project follows an AI-driven development lifecycle supported by Agile-style iteration. The workflow is organised into three phases: Inception, Construction, and Operation.

## Setup

This section prepares imports, output folders, and APIFree API configuration. The notebook requires an APIFree API key for AI generation.

In [ ]:
from pathlib import Path
import json
import os

PROJECT_ROOT = Path.cwd().parent
TASK2_DIR = PROJECT_ROOT / "Task2"
APP_DIR = TASK2_DIR / "app"
DOCS_DIR = TASK2_DIR / "docs"
UML_DIR = TASK2_DIR / "uml"
ASSETS_DIR = APP_DIR / "static" / "images"

for directory in [APP_DIR, DOCS_DIR, UML_DIR, ASSETS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ImportError:
    print("python-dotenv is not installed. The notebook will use existing environment variables only.")

API_KEY = os.getenv("APIFREE_API_KEY")
APIFREE_BASE_URL = os.getenv("APIFREE_BASE_URL", "https://api.apifree.ai/v1")
APIFREE_MODEL = os.getenv("APIFREE_MODEL", "claude-sonnet-4.5")
APIFREE_IMAGE_MODEL = os.getenv("APIFREE_IMAGE_MODEL", "stable-diffusion-xl")

print("APIFree API key detected" if API_KEY else "APIFREE_API_KEY is not configured yet")
print(f"Text model: {APIFREE_MODEL}")
print(f"Image model: {APIFREE_IMAGE_MODEL}")

# Phase 1: Inception

Inception converts the business problem into software development context, including a problem statement, personas, requirements, and user stories.

In [ ]:
business_problem = """
A small travel agency needs a web application that helps tourists generate personalised city travel itineraries. Users should be able to enter a destination, trip length, budget level, and interests. The system should return a clear day-by-day itinerary and present it through both a Flask API and a website.
""".strip()

project_context = {
    "project_name": "Travel Planner Generator",
    "business_problem": business_problem,
    "demo_destination": "Kyoto",
    "demo_days": 3,
    "demo_budget": "medium",
    "demo_interests": ["culture", "food", "nature"],
}

print(project_context["business_problem"])

## Generated SDLC Documentation

This section uses the APIFree API to create the problem statement, personas, requirements, PRD, and user stories.

In [ ]:
def generate_text(task_name, prompt, context):
    """Generate text with the APIFree chat endpoint."""
    api_key = os.getenv("APIFREE_API_KEY")
    base_url = os.getenv("APIFREE_BASE_URL", "https://api.apifree.ai/v1")
    model = os.getenv("APIFREE_MODEL", "claude-sonnet-4.5")

    if not api_key:
        raise RuntimeError("APIFREE_API_KEY is not configured. Add it to the environment before running this notebook.")

    import requests

    response = requests.post(
        f"{base_url.rstrip('/')}/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": model,
            "messages": [
                {"role": "system", "content": "You are a software engineering assistant producing concise coursework artefacts."},
                {"role": "user", "content": prompt},
            ],
            "temperature": 0.3,
        },
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"].strip()

In [ ]:
problem_statement_prompt = f"""
Given the business problem below, write one concise software engineering problem statement.

Business problem:
{business_problem}
""".strip()

personas_prompt = f"""
Generate 3-4 concise user personas for this software project.
Each persona should include the user's role, goal, and main concern.

Business problem:
{business_problem}
""".strip()

requirements_prompt = f"""
Write functional and non-functional requirements for this project in markdown.
Keep the scope suitable for a coursework Flask API and website.

Business problem:
{business_problem}
""".strip()

prd_prompt = f"""
Write a concise Product Requirements Document for this project with headings:
Overview, Goals, Scope, Constraints, Success Criteria.

Business problem:
{business_problem}
""".strip()

user_stories_prompt = f"""
Return only valid JSON for 3-5 Agile user stories using this schema:
{{
  "user_stories": [
    {{
      "id": "US1",
      "role": "...",
      "goal": "...",
      "benefit": "...",
      "acceptance_criteria": ["...", "..."]
    }}
  ]
}}

Business problem:
{business_problem}
""".strip()

generated_docs = {
    "problem_statement": generate_text("problem_statement", problem_statement_prompt, project_context),
    "personas": generate_text("personas", personas_prompt, project_context),
    "requirements": generate_text("requirements", requirements_prompt, project_context),
    "prd": generate_text("prd", prd_prompt, project_context),
    "user_stories": generate_text("user_stories", user_stories_prompt, project_context),
}

for name, content in generated_docs.items():
    suffix = "json" if name == "user_stories" else "md"
    output_path = DOCS_DIR / f"{name}.{suffix}"
    output_path.write_text(content, encoding="utf-8")
    print(f"Saved {output_path.relative_to(PROJECT_ROOT)}")

In [ ]:
print("Problem statement:\n")
print(generated_docs["problem_statement"])

print("\nUser stories preview:\n")
print(generated_docs["user_stories"][:800])

# Phase 2: Construction

Construction will generate UML diagrams, Flask API code, website code, and an automatically generated destination image.

## Generated UML and Application Code

This section generates UML diagrams from the SDLC documentation and renders them through the public PlantUML server.

In [ ]:
import re
import zlib
from urllib.parse import quote


def extract_plantuml(text):
    """Extract PlantUML code from a model response."""
    fenced_match = re.search(r"```(?:plantuml|puml)?\s*(.*?)```", text, re.DOTALL | re.IGNORECASE)
    if fenced_match:
        text = fenced_match.group(1).strip()
    start = text.find("@startuml")
    end = text.find("@enduml")
    if start == -1 or end == -1:
        raise ValueError("The generated UML response does not contain @startuml and @enduml.")
    return text[start:end + len("@enduml")].strip()


def plantuml_encode(text):
    """Encode PlantUML text for the PlantUML server."""
    data = zlib.compress(text.encode("utf-8"))[2:-4]
    alphabet = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz-_"

    def encode_3bytes(bytes_chunk):
        if len(bytes_chunk) < 3:
            bytes_chunk += b"\x00" * (3 - len(bytes_chunk))
        b1, b2, b3 = bytes_chunk
        c1 = b1 >> 2
        c2 = ((b1 & 0x3) << 4) | (b2 >> 4)
        c3 = ((b2 & 0xF) << 2) | (b3 >> 6)
        c4 = b3 & 0x3F
        return alphabet[c1] + alphabet[c2] + alphabet[c3] + alphabet[c4]

    return "".join(encode_3bytes(data[i:i + 3]) for i in range(0, len(data), 3))


def render_plantuml_png(plantuml_text, output_path):
    """Render a PlantUML diagram to PNG using the public PlantUML server."""
    import requests

    encoded = plantuml_encode(plantuml_text)
    url = f"https://www.plantuml.com/plantuml/png/{encoded}"
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    output_path.write_bytes(response.content)
    return url

In [ ]:
uml_context = "\n\n".join([
    "Business Problem:\n" + business_problem,
    "Problem Statement:\n" + generated_docs["problem_statement"],
    "Requirements:\n" + generated_docs["requirements"],
    "User Stories:\n" + generated_docs["user_stories"],
])

uml_prompts = {
    "use_case_diagram": f"""
Generate a UML use case diagram in valid PlantUML for the Travel Planner Generator.
Include actors such as Tourist and Travel Agency Assistant.
Include use cases for entering preferences, generating an itinerary, viewing the website, requesting a destination image, and accessing the API.
Return only PlantUML code from @startuml to @enduml.

{uml_context}
""".strip(),
    "sequence_diagram": f"""
Generate a UML sequence diagram in valid PlantUML for the main user journey.
Show the User submitting travel preferences to the Website, the Website calling the Flask API, the Flask API calling APIFree, and the Website displaying the generated itinerary and image.
Return only PlantUML code from @startuml to @enduml.

""".strip(),
}

generated_uml = {}
for diagram_name, prompt in uml_prompts.items():
    print(f"Generating {diagram_name}...")
    uml_response = generate_text(diagram_name, prompt, project_context)
    plantuml_text = extract_plantuml(uml_response)
    generated_uml[diagram_name] = plantuml_text

    puml_path = UML_DIR / f"{diagram_name}.puml"
    png_path = UML_DIR / f"{diagram_name}.png"
    puml_path.write_text(plantuml_text, encoding="utf-8")
    render_url = render_plantuml_png(plantuml_text, png_path)

    print(f"Saved {puml_path.relative_to(PROJECT_ROOT)}")
    print(f"Saved {png_path.relative_to(PROJECT_ROOT)}")
    print(f"Rendered with {render_url}\n")

In [ ]:
for diagram_name, plantuml_text in generated_uml.items():
    print(f"--- {diagram_name} ---")
    print(plantuml_text[:800])
    print()

In [ ]:
from IPython.display import Image, display

for diagram_name in generated_uml:
    png_path = UML_DIR / f"{diagram_name}.png"
    print(f"Displaying {png_path.relative_to(PROJECT_ROOT)}")
    display(Image(filename=str(png_path)))

# Phase 3: Operation

Operation will prepare tests, Docker support, CI/CD workflow configuration, and deployment evidence for Task2.